# Buril — entrenamiento del limpiador de trazos (Kaggle)

Cuaderno **driver fino**: no copia la lógica, sólo llama a los scripts del repo
(`tools/buril.py`). Así, cuando el entrenamiento o el dataset cambien, este
cuaderno ya los usa en el próximo `git pull`.

**Kaggle ahora entrena; Google Colab queda para probar otros modelos.**

## Cómo dejarlo andando (una vez)

1. **Acelerador**: barra derecha → *Session options* → *Accelerator* → **GPU T4 x2**.
2. **Internet**: activá **Internet** en esas mismas opciones (hace falta para
   clonar el repo y para el buzón de Releases de GitHub).
3. **Secreto**: menú **Add-ons → Secrets → Add secret**, nombre `GH_TOKEN` y como
   valor un PAT fine-grained de GitHub con permiso *Contents: Read and write*
   sobre `LuisAlejandro544/Dibujo-`. Sólo se lee desde el código; no lo escribas
   en una celda.
4. **Correr en segundo plano (clave)**: arriba a la derecha, **Save Version →
   Save & Run All (Commit)**. Un *commit* corre en el servidor **sin timer de
   inactividad**: podés cerrar el navegador y apagar el celular; sigue entrenando.
   (El modo interactivo "Run" sí se corta por inactividad; no lo uses para esto.)

El cuaderno recupera el `.npz` cacheado de la release `dataset-buril`, entrena y
publica el `.tflite` en `modelo-buril` (el buzón que lee `android-apk.yml`).
También sube los puntos de control a `checkpoints-buril` para poder reanudar.

**Versionado**: cambiá `VERSION_DATOS` / `VERSION_MODELO` en la celda 1 para
publicar una versión nueva (por defecto `v1.0.0-experimental.1a`).


In [ ]:
# @title 1. Configuración, clonado y dependencias
REPO = "LuisAlejandro544/Dibujo-"           # @param {type:"string"}
RAMA = "main"                               # @param {type:"string"}
VERSION_DATOS  = "v1.0.0-experimental.1a"   # @param {type:"string"}
VERSION_MODELO = "v1.0.0-experimental.1a"   # @param {type:"string"}
TAG_DATOS  = "dataset-buril"                # @param {type:"string"}
TAG_MODELO = "modelo-buril"                 # @param {type:"string"}
TAG_CHECKPOINTS = "checkpoints-buril"       # @param {type:"string"}
LADO          = 512                         # @param {type:"integer"}
POR_CATEGORIA = 200                         # @param {type:"integer"}
PARCHES       = 8                          # @param {type:"integer"}
BANDA         = 1                           # @param {type:"integer"}
EPOCAS        = 30                          # @param {type:"integer"}
AUMENTOS      = 2                           # @param {type:"integer"}
LR            = 0.002                       # @param {type:"number"}
LOTE          = 256                          # @param {type:"integer"}
LOTE_FINAL    = 128                          # @param {type:"integer"}
FASE_RAPIDA   = 0.75                        # @param {type:"number"}
CALENTAMIENTO = 1                           # @param {type:"integer"}

import os, sys, subprocess

RUTA = "/kaggle/working/Dibujo-"
URL = f"https://github.com/{REPO}.git"
if os.path.isdir(os.path.join(RUTA, ".git")):
    subprocess.run(["git", "fetch", "origin", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "checkout", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", RAMA], cwd=RUTA, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", RAMA, URL, RUTA],
                   check=True)

os.chdir(RUTA)

# Kaggle trae TensorFlow con GPU; si no estuviera, se instala.
try:
    import tensorflow as tf
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "tensorflow"],
                   check=True)
    import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print("Repositorio listo en", RUTA, "| rama", RAMA)
print("TensorFlow", tf.__version__, "| GPU:", [g.name for g in gpus] or "ninguna (CPU)")
if not gpus:
    print("AVISO: no hay GPU. En Session options elegí Accelerator = GPU T4 x2;"
          " si no, el entrenamiento va mucho más lento.")



In [ ]:
# @title 2. Utilidades de GitHub (release como buzón)
import json, os, requests
from kaggle_secrets import UserSecretsClient

try:
    TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
except Exception as exc:
    raise SystemExit(
        "Falta el secreto GH_TOKEN en Kaggle (Add-ons -> Secrets; permiso "
        f"Contents: Read and write). Detalle: {exc}")

CABECERAS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/vnd.github+json"}
API = f"https://api.github.com/repos/{REPO}"


def asegurar_release(tag, nombre, cuerpo):
    """Devuelve la release [tag] creándola si no existe."""
    r = requests.get(f"{API}/releases/tags/{tag}", headers=CABECERAS)
    if r.status_code == 200:
        return r.json()
    r = requests.post(f"{API}/releases", headers=CABECERAS,
                      json={"tag_name": tag, "name": nombre, "body": cuerpo})
    r.raise_for_status()
    return r.json()


def bajar_asset(tag, nombre, destino):
    """Baja el asset [nombre] de la release [tag] a [destino]; True si existía."""
    r = requests.get(f"{API}/releases/tags/{tag}", headers=CABECERAS)
    if r.status_code != 200:
        return False
    for a in r.json().get("assets", []):
        if a["name"] == nombre:
            datos = requests.get(
                a["url"], headers={**CABECERAS, "Accept": "application/octet-stream"})
            datos.raise_for_status()
            os.makedirs(os.path.dirname(destino), exist_ok=True)
            with open(destino, "wb") as f:
                f.write(datos.content)
            print(f"Bajado {nombre} ({len(datos.content)} bytes) de {tag}")
            return True
    return False


def subir_asset(tag, ruta, nombre):
    """Sube (o reemplaza) [ruta] como asset [nombre] de la release [tag]."""
    release = asegurar_release(tag, tag, f"Buzón `{tag}`.")
    for viejo in requests.get(f"{API}/releases/{release['id']}/assets",
                              headers=CABECERAS).json():
        if viejo["name"] == nombre:
            requests.delete(f"{API}/releases/assets/{viejo['id']}", headers=CABECERAS)
    with open(ruta, "rb") as f:
        datos = f.read()
    subida = requests.post(
        f"https://uploads.github.com/repos/{REPO}/releases/{release['id']}/assets",
        headers={**CABECERAS, "Content-Type": "application/octet-stream"},
        params={"name": nombre}, data=datos)
    subida.raise_for_status()
    print(f"Publicado {nombre} ({len(datos)} bytes) en la release {tag}")


def actualizar_release(tag, nombre, cuerpo):
    """Fija el nombre y las notas de la release [tag]."""
    release = asegurar_release(tag, nombre, cuerpo)
    r = requests.patch(f"{API}/releases/{release['id']}", headers=CABECERAS,
                       json={"name": nombre, "body": cuerpo})
    r.raise_for_status()
    print(f"Release {tag} actualizada: {nombre}")


In [ ]:
# @title 3. Dataset .npz: recuperar de la release o prepararlo
DIR = ".cache/limpiar_trazos_planb/procesado"
NPZ = f"{DIR}/pares_esq_lado{LADO}.npz"
MANIF = f"{DIR}/manifiesto_esq_lado{LADO}.json"
ASSET_NPZ = f"pares_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.npz"
ASSET_MANIF = f"manifiesto_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.json"

asegurar_release(TAG_DATOS, "Dataset Buril (cache del .npz)",
                 "Cache del dataset comercial (Quick, Draw!, CC BY 4.0) que usa "
                 "`tools/buril.py`. Lo llenan `tools/buril_kaggle.ipynb` y "
                 "`tools/buril_colab.ipynb` para no preprocesar en cada sesión.")

if bajar_asset(TAG_DATOS, ASSET_NPZ, NPZ):
    bajar_asset(TAG_DATOS, ASSET_MANIF, MANIF)
    print(f"Dataset recuperado de la release {TAG_DATOS} ({VERSION_DATOS}).")
else:
    print("No hay cache para esta versión; preparando el dataset (puede tardar)...")
    subprocess.run([sys.executable,
                    "tools/preparar_dataset_limpiar_trazos_planb.py", "todo",
                    "--lado", str(LADO), "--por-categoria", str(POR_CATEGORIA),
                    "--version", VERSION_DATOS], check=True)
    subir_asset(TAG_DATOS, NPZ, ASSET_NPZ)
    if os.path.exists(MANIF):
        subir_asset(TAG_DATOS, MANIF, ASSET_MANIF)

print("npz listo:", NPZ, os.path.getsize(NPZ), "bytes")


In [ ]:
# @title 4. Puntos de control: recuperar de la release (reanudar)
CKPT = os.path.join(RUTA, ".cache/modelos/checkpoints")
os.makedirs(CKPT, exist_ok=True)

# Si hay un entrenamiento a medias en `checkpoints-buril`, se reanuda. Un commit
# de Kaggle tiene tope de ~9-12 h y no se corta por inactividad, pero esto cubre
# reinicios manuales o corridas largas.
if bajar_asset(TAG_CHECKPOINTS, "estado.json", os.path.join(CKPT, "estado.json")):
    for nombre in ["ultimo.weights.h5", "mejor.weights.h5"]:
        bajar_asset(TAG_CHECKPOINTS, nombre, os.path.join(CKPT, nombre))
else:
    print("No hay puntos de control previos; se entrena de cero.")


In [ ]:
# @title 5. Entrenar Buril (rápido y con pulido final)
SALIDA = os.path.join(RUTA, ".cache/modelos/limpiar_trazos.tflite")

BASE = [sys.executable, "tools/buril.py",
        "--pares", NPZ,
        "--salida", SALIDA,
        "--epocas", str(EPOCAS),
        "--aumentos", str(AUMENTOS),
        "--banda", str(BANDA),
        "--lr", str(LR),
        "--parches-por-imagen", str(PARCHES),
        "--lote", str(LOTE),
        "--lote-final", str(LOTE_FINAL),
        "--fase-rapida", str(FASE_RAPIDA),
        "--calentamiento", str(CALENTAMIENTO),
        "--validacion-cada", "2",
        "--cldice-iteraciones", "5",
        "--pasos-por-ejecucion", "2",
        "--distribucion", "auto",
        "--checkpoints", CKPT,
        "--nombre", "Buril", "--version", VERSION_MODELO,
        "--reanudar", CKPT]
nuevo = os.path.join(CKPT, "estado.json")
if os.path.exists(nuevo):
    with open(nuevo, encoding="utf-8") as f:
        print("Reanudando desde la época", json.load(f)["epoca_completadas"])

# Sube checkpoints cada ~2 min mientras entrena: si la sesión muere antes del
# final, `checkpoints-buril` ya tiene el último estado para reanudar.
import threading, time, shutil

parar = threading.Event()


def _subir_periodico(segundos=120):
    temporal = os.path.join("/kaggle/working", ".ckpt_subida")
    os.makedirs(temporal, exist_ok=True)
    while not parar.wait(segundos):
        for nombre in ["estado.json", "ultimo.weights.h5", "mejor.weights.h5"]:
            ruta = os.path.join(CKPT, nombre)
            if not os.path.exists(ruta):
                continue
            copia = os.path.join(temporal, nombre)
            try:
                shutil.copy2(ruta, copia)
                subir_asset(TAG_CHECKPOINTS, copia, nombre)
            except Exception as exc:  # noqa: BLE001
                print("checkpoint intermedio no se pudo subir:", exc)


hilo = threading.Thread(target=_subir_periodico, daemon=True)
hilo.start()

# Cascada de respaldo: primero lo más rápido (mixta + XLA + 2 GPUs); si algo no
# está soportado, cae a configuraciones más conservadoras en vez de morir.
INTENTOS = [
    ["--precision", "auto", "--xla", "auto"],
    ["--precision", "float32", "--xla", "auto"],
    ["--precision", "float32", "--xla", "no"],
    ["--distribucion", "uno", "--precision", "float32", "--xla", "no"],
]
entrenado = False
try:
    for flags in INTENTOS:
        print("\n==== Entrenando con", flags, "====", flush=True)
        codigo = subprocess.run(BASE + flags, cwd=RUTA).returncode
        if codigo == 0:
            entrenado = True
            break
        print(f"Falló con {flags} (código {codigo}); probando una configuración "
              "más conservadora...", flush=True)
finally:
    parar.set()
if not entrenado:
    raise SystemExit("El entrenamiento falló con todas las configuraciones.")

# Copia de cortesía en la raíz de la salida del cuaderno (para descargar fácil).
shutil.copy(SALIDA, os.path.join("/kaggle/working", os.path.basename(SALIDA)))
print("Modelo entrenado:", SALIDA, os.path.getsize(SALIDA), "bytes")



In [ ]:
# @title 6. Publicar modelo y puntos de control en las releases
from os.path import basename, splitext

CUERPO = "\n\n".join([
    f"Modelo **Buril {VERSION_MODELO}** (`limpiar_trazos.tflite`, limpiador de "
    "trazos float16) entrenado en Kaggle con `tools/buril_kaggle.ipynb`.",
    f"- Dataset: `{ASSET_NPZ}` (release `{TAG_DATOS}`).",
    f"- Épocas: {EPOCAS} · aumentos: {AUMENTOS} · parches/hoja: {PARCHES}.",
    f"- Schedule: lote {LOTE} (fase rápida) -> {LOTE_FINAL} (pulido), "
    f"calentamiento {CALENTAMIENTO}, LR {LR} (coseno).",
    f"- Objetivo: banda de {2 * BANDA + 1} px · pérdida BCE + Dice + clDice.",
    "- `android-apk.yml` baja este asset y lo inyecta en el APK.",
])
actualizar_release(TAG_MODELO, f"Modelo Buril {VERSION_MODELO}", CUERPO)

for archivo in [SALIDA, splitext(SALIDA)[0] + ".json"]:
    if os.path.exists(archivo):
        subir_asset(TAG_MODELO, archivo, basename(archivo))

# Puntos de control (pesos + estado) para poder reanudar en otra sesión.
for nombre in ["estado.json", "ultimo.weights.h5", "mejor.weights.h5"]:
    ruta = os.path.join(CKPT, nombre)
    if os.path.exists(ruta):
        subir_asset(TAG_CHECKPOINTS, ruta, nombre)

with open(splitext(SALIDA)[0] + ".json", encoding="utf-8") as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))


## Siguiente paso

La release `modelo-buril` ya tiene el `.tflite` de la versión pedida y el `.npz`
quedó cacheado en `dataset-buril`. Para armar el APK con este modelo, andá a
*Actions → "APK Android (assembleDebug)" → Run workflow*; `android-apk.yml` baja
el `.tflite` de la release y lo inyecta en los assets antes de `assembleDebug`.

Si querés **reentrenar desde cero** (ignorar los checkpoints), subí `VERSION_MODELO`
o borrá los assets de `checkpoints-buril` y volvé a correr el commit.

> Este cuaderno es un driver fino: la lógica de entrenamiento vive en
> `tools/buril.py`. Si necesitás cambiar el modelo o la pérdida, editá ese script
> (y corré `python3 tools/buril.py` localmente o vía Actions).
